If you want to type along with me, head to [this notebook](https://humboldt.cloudbank.2i2c.cloud/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2Fmlekimchi%2Fdata111_fall26&branch=main&urlpath=tree%2Fdata111_fall26%2Flectures%2Flec19_live.ipynb) instead. If you prefer follow along by executing the cells, stay in this notebook.

# Lecture 19: P-values and A/B Testing
v.ekc

Two halves today: first, what a p-value *really* is (and how often it fools you) — then **A/B testing**, comparing two groups with the permutation test. (Slides: KL Lecture 19 deck.)

**Today's sections:**
1. The instructor's defense, recapped
2. Try It — 2000 coins
3. Comparing two samples
4. Random permutation (shuffling)
5. The permutation test

In [ ]:
from datascience import *
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')

---
## 1. The Instructor's Defense, Recapped

The full test from last time in one pass — watch for the two p-value computations at the end (`np.count_nonzero` and `np.mean` of a bool array are the same trick).

In [ ]:
scores = Table.read_table('scores_by_section.csv')
scores

In [ ]:
scores.group('Section')

In [ ]:
scores.group('Section', np.average).show()

In [ ]:
observed_average = 13.6667 

In [ ]:
random_sample = scores.sample(27, with_replacement=False)
random_sample

In [ ]:
np.average(random_sample.column('Midterm'))

In [ ]:
# Simulate one value of the test statistic 
# under the hypothesis that the section is like a random sample from the class
def random_sample_midterm_avg():
    random_sample = scores.sample(27, with_replacement = False)
    return np.average(random_sample.column('Midterm'))

In [ ]:
# Simulate 50,000 copies of the test statistic

sample_averages = make_array()

for i in np.arange(50000):
    sample_averages = np.append(sample_averages, random_sample_midterm_avg())    

## Our Decision

In [ ]:
# Compare the simulated distribution of the statistic
# and the actual observed statistic
averages_tbl = Table().with_column('Random Sample Average', sample_averages)
averages_tbl.hist(bins = 20)
plots.scatter(observed_average, -0.01, color='red', s=120);

### Approach 1

In [ ]:
# (1) Calculate the p-value: simulation area beyond observed value
np.count_nonzero(sample_averages <= observed_average) / len(sample_averages)
# (2) See if this is less than 5%

In [ ]:
# Another way 
np.mean(sample_averages <= observed_average)

### Approach 2

In [ ]:
# (1) Find simulated value corresponding to 5% of 50,000 = 2500
five_percent_point = averages_tbl.sort(0).column(0).item(int(len(sample_averages)*0.05))
five_percent_point

In [ ]:
# (2) See if this value is greater than observed value
observed_average

### Visual Representation

In [ ]:
averages_tbl.hist(bins = 20)
plots.plot([five_percent_point, five_percent_point], [0, 0.35], color='gold', lw=2)
plots.title('Area to the left of the gold line: 5%');
plots.scatter(observed_average, -0.01, color='red', s=120);

---
### Try It 1: 2000 coins 🪙 (the big one)

2000 people each toss a fair coin 100 times and run a hypothesis test:

- **Null:** the coin is fair. **Alternative:** it isn't.
- **Test statistic:** |number of heads − 50|. **Cutoff:** p < 0.05.

All 2000 coins really are fair. **About how many people will wrongly conclude their coin is unfair?** Fill in the cells to find out.

In [ ]:
# Fill in — this cell errors until you do! 😅
num_people =
num_tosses = 
fair_coin = 

In [ ]:
# Fill in — this cell errors until you do! 😅
def get_one_statistic():
    num_heads_in_100_tosses = 
    return 

In [ ]:
# Fill in — this cell errors until you do! 😅
# Generate statistics for the 2000 people in the room
peoples_statistics = 

for :
    peoples_statistics = 

In [ ]:
# Fill in — this cell errors until you do! 😅
# Now perform the hypothesis test (simulate 10000 test statistics under the null hypothesis)
# NOTE, we will just do this once. All 2000 people should end up with similar empirical 
# distributions of the test statistic
simulated_statistics =

for i in np.arange(10000):
    simulated_statistics = 

In [ ]:
# Fill in — this cell errors until you do! 😅
# Plot the empirical distribution of test statistic
stats_table = 


In [ ]:
# Fill in — this cell errors until you do! 😅
# Get each person's P-value 
p_vals = 
for persons_stat in peoples_statistics:
    p_vals= 
    
p_vals

In [ ]:
# How many of the 2000 get a p-value below 0.05?


<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*Everything is the simulation recipe — the punchline is the last line.*

```python
num_people = 2000
num_tosses = 100
fair_coin = make_array('heads', 'tails')

def get_one_statistic():
    num_heads_in_100_tosses = sum(np.random.choice(fair_coin, num_tosses) == 'heads')
    return abs(num_heads_in_100_tosses - 50)

# Statistics for the 2000 people in the room
peoples_statistics = make_array()
for i in np.arange(num_people):
    peoples_statistics = np.append(peoples_statistics, get_one_statistic())

# Null distribution — simulated once, shared by everyone
simulated_statistics = make_array()
for i in np.arange(10000):
    simulated_statistics = np.append(simulated_statistics, get_one_statistic())

stats_table = Table().with_column('|Heads - 50|', simulated_statistics)
stats_table.hist()

# Each person's p-value
p_vals = make_array()
for persons_stat in peoples_statistics:
    p_vals = np.append(p_vals, np.mean(simulated_statistics >= persons_stat))

sum(p_vals <= 0.05)     # ≈ 100 people!
```

</details>

> **The moral** (KL deck, slides 12–16): with a 5% cutoff, about 5% of *true* nulls get rejected — here, ~100 innocent coins condemned. A p-value cutoff is an **error rate you're agreeing to**, not a truth detector. Remember this every time you read "statistically significant."


---
## 2. Comparing Two Samples (A/B Testing)

New question type: do babies of smoking mothers weigh less? Two groups — smokers and non-smokers. Are the two distributions *really* different, or is the gap just sampling noise? (KL deck, slides 18–19.)

In [ ]:
births = Table.read_table('baby.csv')

In [ ]:
births

In [ ]:
smoking_and_birthweight = births.select('Maternal Smoker', 'Birth Weight')

In [ ]:
smoking_and_birthweight.group('Maternal Smoker')

In [ ]:
smoking_and_birthweight.hist('Birth Weight', group='Maternal Smoker')

# Test Statistic

[Question] What values of our statistic are in favor of the alternative: positive or negative?

In [ ]:
means_table = smoking_and_birthweight.group('Maternal Smoker', np.average)
means_table

In [ ]:
means = means_table.column(1)
observed_difference = means.item(1) - means.item(0)
observed_difference

In [ ]:
def difference_of_means(table, label, group_label):
    """Takes: name of table, column label of numerical variable,
    column label of group-label variable
    Returns: Difference of means of the two groups"""
    
    #table with the two relevant columns
    reduced = table.select(label, group_label)  
    
    # table containing group means
    means_table = reduced.group(group_label, np.average)
    # array of group means
    means = means_table.column(1)
    
    return means.item(1) - means.item(0)

In [ ]:
difference_of_means(births, 'Birth Weight', 'Maternal Smoker')

---
## 3. Random Permutation (Shuffling)

**The null says the label doesn't matter** — so let's literally shuffle the labels and see what differences pure chance produces. `sample(with_replacement=False)` is a shuffle.

In [ ]:
letters = Table().with_column('Letter', make_array('a', 'b', 'c', 'd', 'e'))

In [ ]:
letters.sample()

In [ ]:
letters.sample(with_replacement = False)

In [ ]:
letters.with_column('Shuffled', letters.sample(with_replacement = False).column(0))

---
## 4. Simulation Under the Null

Attach shuffled labels, recompute the difference of means — that's one draw from the null world.

In [ ]:
smoking_and_birthweight

In [ ]:
shuffled_labels = smoking_and_birthweight.sample(with_replacement=False).column('Maternal Smoker')

In [ ]:
original_and_shuffled = smoking_and_birthweight.with_column(
    'Shuffled Label', shuffled_labels
)

In [ ]:
original_and_shuffled

In [ ]:
difference_of_means(original_and_shuffled, 'Birth Weight', 'Shuffled Label')

In [ ]:
difference_of_means(original_and_shuffled, 'Birth Weight', 'Maternal Smoker')

---
## 5. The Permutation Test

Repeat the shuffle 2500 times and compare the observed difference to the shuffled ones:

In [ ]:
def one_simulated_difference(table, label, group_label):
    """Takes: name of table, column label of numerical variable,
    column label of group-label variable
    Returns: Difference of means of the two groups after shuffling labels"""
    
    # array of shuffled labels
    shuffled_labels = table.sample(with_replacement = False).column(group_label)
    
    # table of numerical variable and shuffled labels
    shuffled_table = table.select(label).with_column('Shuffled Label', shuffled_labels)
    
    return difference_of_means(shuffled_table, label, 'Shuffled Label')   

In [ ]:
one_simulated_difference(births, 'Birth Weight', 'Maternal Smoker')

In [ ]:
differences = make_array()

for i in np.arange(2500):
    new_difference = one_simulated_difference(births, 'Birth Weight', 'Maternal Smoker')
    differences = np.append(differences, new_difference)

In [ ]:
Table().with_column('Difference Between Group Means', differences).hist()
print('Observed Difference:', observed_difference)
plots.title('Prediction Under the Null Hypothesis');

> **Read the picture:** the observed difference (−9.27 oz) is nowhere near the shuffled differences (all within ±5). The null — "smoking label doesn't matter" — is untenable. Lab 7 (Great British Bake Off) is this exact test with cake. 🍰

---
> **Today's takeaway:** a p-value cutoff is an error rate; and to compare two groups, shuffle the labels — if the observed gap dwarfs the shuffled gaps, the groups genuinely differ.

## Appendix — Quick Reference

Full `datascience` docs: [data8.org/datascience](https://data8.org/datascience/)

### Minimal template — permutation test
```python
shuffled = t.sample(with_replacement=False).column(group_label)
one_diff = difference_of_means(t.with_column('Shuffled', shuffled), value_label, 'Shuffled')
```